# Lab 12 — Tool-grounded single-agent

Use case: support extraction for Ukrainian admin/service support messages.

## 1. Install deps

In [1]:
import sys
import subprocess

if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', '/content/nlp/project_lab12/requirements.txt'])
'deps ready'

'deps ready'

## 2. Data / test cases

In [2]:
import json
import sys
from pathlib import Path

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    extra = [Path('/content/nlp'), Path('/content'), Path(r'C:/Users/maia1/data/politiekh/masters/nlp')]
    for candidate in candidates + extra:
        if (candidate / 'project_lab12').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root containing project_lab12')

ROOT = find_project_root()
PROJ = ROOT / 'project_lab12'
SRC = PROJ / 'src'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

CASES_PATH = PROJ / 'data' / 'sample' / 'test_cases_lab12.jsonl'
with CASES_PATH.open('r', encoding='utf-8') as handle:
    CASES = [json.loads(line) for line in handle if line.strip()]
len(CASES), CASES[0]['case_id']

(12, 'case_001_simple_compensation')

## 3. Tool definitions

In [3]:
import pandas as pd
from tools import classify_issue_type, extract_support_fields, validate_required_fields

demo_text = CASES[3]['input']
{
    'classify_issue_type': classify_issue_type(demo_text),
    'extract_support_fields': extract_support_fields(demo_text),
    'validate_required_fields': validate_required_fields(extract_support_fields(demo_text)),
}

{'classify_issue_type': {'issue_type': 'payment_or_amount',
  'route_hint': 'payment_route',
  'ambiguous': False,
  'noisy': False,
  'reason': 'money or compensation cues'},
 'extract_support_fields': {'primary_service': 'єВідновлення',
  'services_mentioned': ['єВідновлення'],
  'issue_type': 'payment_or_amount',
  'document_type': None,
  'amounts_uah': [500],
  'date_text': 'завтра',
  'location_text': None},
 'validate_required_fields': {'valid': True,
  'missing_fields': [],
  'schema_errors': [],
  'warnings': ['relative date needs human interpretation']}}

## 4. Tool call logger

In [4]:
from tool_logger import ToolCallLogger

logger = ToolCallLogger()
logger.call('demo', 'classify_issue_type', classify_issue_type, text=demo_text, reason='demo call')
pd.DataFrame(logger.records)

,timestamp,task_id,tool_name,input,output,success,error,reason,metadata
0,2026-05-16T12:19:03.407846+00:00,demo,classify_issue_type,{'text': 'Переказати 500 грн на ремонт завтра ...,"{'issue_type': 'payment_or_amount', 'route_hin...",True,None,demo call,{}


## 5. Agent design

In [5]:
from agent import BaselineSupportAssistant, ToolGroundedSupportAgent
from eval_agent import compare_outputs, compute_metrics, summarize_case

baseline = BaselineSupportAssistant()
tool_logger = ToolCallLogger()
agent = ToolGroundedSupportAgent(logger=tool_logger)
baseline, agent

(BaselineSupportAssistant(),
 ToolGroundedSupportAgent(logger=ToolCallLogger(records=[])))

## 6. Baseline LLM without tools

In [6]:
baseline_results = {case['case_id']: baseline.run(case['case_id'], case['input']) for case in CASES}
baseline_preview = []
for case in CASES[:3]:
    baseline_eval = compare_outputs(case['expected'], baseline_results[case['case_id']]['final_output'])
    baseline_preview.append({
        'case_id': case['case_id'],
        'baseline_route': baseline_results[case['case_id']]['route'],
        'baseline_rating': baseline_eval['rating'],
        'baseline_output': baseline_results[case['case_id']]['final_output'],
    })
pd.DataFrame(baseline_preview)

,case_id,baseline_route,baseline_rating,baseline_output
0,case_001_simple_compensation,application_submission_route,partly,"{'primary_service': 'єВідновлення', 'services_..."
1,case_002_document_cnap,document_requirement_route,partly,"{'primary_service': 'єВідновлення', 'services_..."
2,case_003_ambiguous_service,application_submission_route,partly,"{'primary_service': 'ЦНАП', 'services_mentione..."


## 7. Agent with tools

In [7]:
tool_results = {case['case_id']: agent.run(case['case_id'], case['input']) for case in CASES}
preview_rows = []
for case in CASES[:5]:
    result = tool_results[case['case_id']]
    evaluation = compare_outputs(case['expected'], result['final_output'])
    preview_rows.append({
        'case_id': case['case_id'],
        'route': result['route'],
        'used_tools': ', '.join(result['used_tools']),
        'rating': evaluation['rating'],
        'needs_manual_check': result['needs_manual_check'],
    })
pd.DataFrame(preview_rows)

,case_id,route,used_tools,rating,needs_manual_check
0,case_001_simple_compensation,generic_route,"classify_issue_type, extract_support_fields",partly,False
1,case_002_document_cnap,document_route,"classify_issue_type, extract_support_fields, v...",correct,False
2,case_003_ambiguous_service,generic_route,"classify_issue_type, extract_support_fields",partly,False
3,case_004_relative_date,payment_route,"classify_issue_type, extract_support_fields, v...",correct,False
4,case_005_empty_service_result,payment_route,"classify_issue_type, extract_support_fields, v...",correct,False


## 8. Run 10+ test cases

In [8]:
results_table = []
for case in CASES:
    baseline_eval = compare_outputs(case['expected'], baseline_results[case['case_id']]['final_output'])
    tool_eval = compare_outputs(case['expected'], tool_results[case['case_id']]['final_output'])
    results_table.append({
        'case_id': case['case_id'],
        'tags': ', '.join(case['tags']),
        'baseline_rating': baseline_eval['rating'],
        'tool_rating': tool_eval['rating'],
        'used_tools': ', '.join(tool_results[case['case_id']]['used_tools']),
        'needs_manual_check': tool_results[case['case_id']]['needs_manual_check'],
    })
results_df = pd.DataFrame(results_table)
results_df

,case_id,tags,baseline_rating,tool_rating,used_tools,needs_manual_check
0,case_001_simple_compensation,"simple, final answer references tool output",partly,partly,"classify_issue_type, extract_support_fields",False
1,case_002_document_cnap,"simple, two tools in a row",partly,correct,"classify_issue_type, extract_support_fields, v...",False
2,case_003_ambiguous_service,"ambiguous case, validator finds problem",partly,partly,"classify_issue_type, extract_support_fields",False
3,case_004_relative_date,"relative date, two tools in a row",partly,correct,"classify_issue_type, extract_support_fields, v...",False
4,case_005_empty_service_result,"tool returns empty result, hallucination-prone...",partly,correct,"classify_issue_type, extract_support_fields, v...",False
5,case_006_noisy_typos,"noisy text, validator finds problem",wrong,partly,"classify_issue_type, extract_support_fields, v...",False
6,case_007_passport_queue,"agent should not call extra tool, unnecessary ...",partly,correct,"classify_issue_type, extract_support_fields, v...",False
7,case_008_notary_date,simple,partly,correct,"classify_issue_type, extract_support_fields",False
8,case_009_missing_data,"missing data, validator finds problem",wrong,partly,"classify_issue_type, extract_support_fields, v...",False
9,case_010_manual_review_candidate,"ambiguous case, tool not helpful enough",wrong,partly,"classify_issue_type, extract_support_fields, v...",False


## 9. Tool call logs

In [9]:
LOG_PATH = PROJ / 'docs' / 'tool_logs_lab12.jsonl'
tool_logger.save_jsonl(LOG_PATH)
logs_df = pd.DataFrame(tool_logger.records)
logs_df.head(10)

,timestamp,task_id,tool_name,input,output,success,error,reason,metadata
0,2026-05-16T12:19:03.508357+00:00,case_001_simple_compensation,classify_issue_type,{'text': 'Через Дію подав заяву на єВідновленн...,"{'issue_type': 'application_submission', 'rout...",True,None,decide route and whether later validation is n...,{}
1,2026-05-16T12:19:03.508357+00:00,case_001_simple_compensation,extract_support_fields,{'text': 'Через Дію подав заяву на єВідновленн...,"{'primary_service': 'єВідновлення', 'services_...",True,None,build structured support extraction,{}
2,2026-05-16T12:19:03.508357+00:00,case_002_document_cnap,classify_issue_type,{'text': 'Для єВідновлення у ЦНАПі потрібні до...,"{'issue_type': 'document_requirement', 'route_...",True,None,decide route and whether later validation is n...,{}
3,2026-05-16T12:19:03.508357+00:00,case_002_document_cnap,extract_support_fields,{'text': 'Для єВідновлення у ЦНАПі потрібні до...,"{'primary_service': 'єВідновлення', 'services_...",True,None,build structured support extraction,{}
4,2026-05-16T12:19:03.508357+00:00,case_002_document_cnap,validate_required_fields,"{'data': {'primary_service': 'єВідновлення', '...","{'valid': True, 'missing_fields': [], 'schema_...",True,None,ambiguous service or routing,{'unnecessary': False}
5,2026-05-16T12:19:03.508357+00:00,case_003_ambiguous_service,classify_issue_type,"{'text': 'Якщо подала через ЦНАП, чи треба ще ...","{'issue_type': 'application_submission', 'rout...",True,None,decide route and whether later validation is n...,{}
6,2026-05-16T12:19:03.508357+00:00,case_003_ambiguous_service,extract_support_fields,"{'text': 'Якщо подала через ЦНАП, чи треба ще ...","{'primary_service': 'ЦНАП', 'services_mentione...",True,None,build structured support extraction,{}
7,2026-05-16T12:19:03.508357+00:00,case_004_relative_date,classify_issue_type,{'text': 'Переказати 500 грн на ремонт завтра ...,"{'issue_type': 'payment_or_amount', 'route_hin...",True,None,decide route and whether later validation is n...,{}
8,2026-05-16T12:19:03.508357+00:00,case_004_relative_date,extract_support_fields,{'text': 'Переказати 500 грн на ремонт завтра ...,"{'primary_service': 'єВідновлення', 'services_...",True,None,build structured support extraction,{}
9,2026-05-16T12:19:03.508357+00:00,case_004_relative_date,validate_required_fields,"{'data': {'primary_service': 'єВідновлення', '...","{'valid': True, 'missing_fields': [], 'schema_...",True,None,high-value fields need validation,{'unnecessary': False}


## 10. Metrics

In [10]:
metrics = compute_metrics(CASES, baseline_results, tool_results, tool_logger.records)
metrics_df = pd.DataFrame([
    {'metric': key, 'value': value if not isinstance(value, dict) else json.dumps(value, ensure_ascii=False)}
    for key, value in metrics.items()
])
metrics_df

,metric,value
0,tool_call_success_rate,1.0
1,average_tool_calls_per_task,2.667
2,tasks_with_useful_tool_use,11
3,unnecessary_tool_call_count,1
4,final_answer_ratings,"{""partly"": 5, ""correct"": 7}"
5,tool_error_rate,0.0
6,tasks_using_both_main_tools_rate,1.0
7,tool_output_ignored_rate,0.25
8,final_answer_contradicts_tool_output_rate,0.25


## 11. Error analysis

In [11]:
analysis_rows = []
for case in CASES:
    task_logs = [record for record in tool_logger.records if record['task_id'] == case['case_id']]
    analysis_rows.append(summarize_case(case, tool_results[case['case_id']], task_logs))
analysis_df = pd.DataFrame(analysis_rows)
analysis_df[['case_id', 'error_category', 'rating', 'actual_tool_calls', 'possible_fix']]

,case_id,error_category,rating,actual_tool_calls,possible_fix
0,case_001_simple_compensation,partly correct tool-grounded answer,partly,"[classify_issue_type, extract_support_fields]",Use better post-processing so the agent consum...
1,case_002_document_cnap,showcase,correct,"[classify_issue_type, extract_support_fields, ...",No fix needed.
2,case_003_ambiguous_service,partly correct tool-grounded answer,partly,"[classify_issue_type, extract_support_fields]",Use better post-processing so the agent consum...
3,case_004_relative_date,showcase,correct,"[classify_issue_type, extract_support_fields, ...",No fix needed.
4,case_005_empty_service_result,showcase,correct,"[classify_issue_type, extract_support_fields, ...",No fix needed.
5,case_006_noisy_typos,partly correct tool-grounded answer,partly,"[classify_issue_type, extract_support_fields, ...",Use better post-processing so the agent consum...
6,case_007_passport_queue,unnecessary tool call,correct,"[classify_issue_type, extract_support_fields, ...",Tighten the validation heuristic so low-risk q...
7,case_008_notary_date,"tool not called, although maybe optional",correct,"[classify_issue_type, extract_support_fields]",Document why the agent safely skipped validati...
8,case_009_missing_data,partly correct tool-grounded answer,partly,"[classify_issue_type, extract_support_fields, ...",Use better post-processing so the agent consum...
9,case_010_manual_review_candidate,partly correct tool-grounded answer,partly,"[classify_issue_type, extract_support_fields, ...",Use better post-processing so the agent consum...


## 12. Generate docs

In [12]:
def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.strip() + '\n', encoding='utf-8')

best_cases = analysis_df[analysis_df['rating'] == 'correct']['case_id'].head(3).tolist()
problem_cases = analysis_df[analysis_df['rating'] != 'correct']['case_id'].head(3).tolist()

audit_summary = f"""# Lab 12 audit summary

- Use case: support extraction single-agent with tool grounding.
- Tools: classify_issue_type, extract_support_fields, validate_required_fields.
- Test cases: {len(CASES)}
- Tool call success rate: {metrics['tool_call_success_rate']:.3f}
- Average tool calls per task: {metrics['average_tool_calls_per_task']:.3f}
- Tasks with useful tool use: {metrics['tasks_with_useful_tool_use']}
- Unnecessary tool call count: {metrics['unnecessary_tool_call_count']}
- Final answer ratings: {json.dumps(metrics['final_answer_ratings'], ensure_ascii=False)}
- Best examples: {', '.join(best_cases)}.
- Problem cases: {', '.join(problem_cases)}.
- Next fix: better ambiguity handling and fewer unnecessary validator calls on low-risk cases.
"""

agent_notes = f"""# Lab 12 agent notes

1. Use case: support extraction for Ukrainian admin/service messages.
2. Agent task: route the request, extract structured fields, and decide whether manual check is needed.
3. Tools: classify_issue_type, extract_support_fields, validate_required_fields.
4. Tool selection policy: the agent always calls classifier and extractor, then conditionally calls validator for ambiguous, noisy, document, or payment-heavy cases.
5. Logging: every tool call is written to JSONL with timestamp, task_id, tool name, input, output, success, error, reason, and metadata.
6. What tools improved: better structure, lower hallucination risk, and explicit handling of ambiguity.
7. Where tools were unnecessary: one low-risk passport queue case still triggered validator as an avoidable double-check.
8. Remaining errors: ambiguous multi-service inputs and partly correct outputs on underspecified cases.
9. Next fix: add a small repair step or a stronger ambiguity policy instead of relying on validator alone.
"""

dataset_card = f"""# Dataset card update for Lab 12

- Tested agent use case: single-agent support extraction.
- Input types in test cases: simple requests, missing data, noisy text, ambiguous service mentions, relative dates, and amount-heavy payment cases.
- Tools used on these inputs: issue classification, structured field extraction, required-field validation.
- Noisy / ambiguous cases: yes, explicitly included in the evaluation set.
- Did tool grounding help: yes, especially for structure, ambiguity control, and final answers that cite extracted fields.
"""

readme = f"""# Lab 12

1. Use case: support extraction.
2. Agent task: route a support/admin-service message, extract structured fields, and decide whether manual review is needed.
3. Tools: classify_issue_type, extract_support_fields, validate_required_fields.
4. Run: execute the notebook `project_lab12/notebooks/lab12_tool_grounded_single_agent.ipynb` with Run all.
5. Logs: `project_lab12/docs/tool_logs_lab12.jsonl`.
6. Test cases: `project_lab12/data/sample/test_cases_lab12.jsonl`.
7. Metrics: tool call success rate, average tool calls per task, useful tool-use count, unnecessary tool-call count, final answer ratings.
8. Main conclusion: tool grounding makes the single-agent pipeline more structured and auditable than the no-tool baseline, but ambiguity handling still needs improvement.
"""

write_text(PROJ / 'docs' / 'audit_summary_lab12.md', audit_summary)
write_text(PROJ / 'docs' / 'agent_notes_lab12.md', agent_notes)
write_text(PROJ / 'docs' / 'dataset_card.md', dataset_card)
write_text(PROJ / 'labs' / 'lab12' / 'README.md', readme)

analysis_export = analysis_df[['case_id', 'input', 'expected_behavior', 'actual_tool_calls', 'final_answer', 'error_category', 'possible_fix']]
analysis_export.to_json(PROJ / 'docs' / 'error_cases_lab12.json', orient='records', force_ascii=False, indent=2)

{'audit_summary': str(PROJ / 'docs' / 'audit_summary_lab12.md'), 'log_path': str(LOG_PATH)}

{'audit_summary': 'C:\\Users\\maia1\\data\\politiekh\\masters\\nlp\\project_lab12\\docs\\audit_summary_lab12.md',
 'log_path': 'C:\\Users\\maia1\\data\\politiekh\\masters\\nlp\\project_lab12\\docs\\tool_logs_lab12.jsonl'}